In [4]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Create sample data since the CSV files don't exist
print("Creating sample datasets...")

# Create sample South African vendors dataset
np.random.seed(42)  # For reproducible results
vendors_data = {
    'Vendor_ID': [f'V{str(i).zfill(3)}' for i in range(1, 21)],
    'Vendor_Name': [f'Company_{i}' for i in range(1, 21)],
    'Bank_Account': ['12345678', '87654321', '12345678', '11111111', '22222222'] * 4,  # Some duplicates for testing
    'Risk_Score': np.random.randint(1, 10, 20)
}
vendors_df = pd.DataFrame(vendors_data)

# Create sample procurement invoice ledger
invoice_data = {
    'Invoice_ID': [f'INV{str(i).zfill(4)}' for i in range(1, 51)],
    'Vendor_ID': np.random.choice(vendors_df['Vendor_ID'], 50),
    'Amount_ZAR': np.random.choice([1500.00, 2500.00, 1500.00, 3000.00], 50),  # Some duplicates for testing
    'Transaction_Date': [datetime.now() - timedelta(days=np.random.randint(1, 365)) for _ in range(50)],
    'Approver': np.random.choice(['Manager_A', 'Manager_B', 'Manager_C'], 50)
}
ledger_df = pd.DataFrame(invoice_data)

print("Sample data created successfully!")

# --- 1. RUN ENGINE: OPTION A (COLLUSION & VELOCITY DETECTION) ---

# Detect Shared Bank Accounts across distinct Vendor IDs
shared_bank_mask = vendors_df.duplicated(subset=['Bank_Account'], keep=False)
flagged_vendors = vendors_df[shared_bank_mask].copy()

# Detect Duplicate Transactions (Same Vendor, Same Amount)
ledger_df['Transaction_Date'] = pd.to_datetime(ledger_df['Transaction_Date'])
ledger_sorted = ledger_df.sort_values(by=['Vendor_ID', 'Amount_ZAR', 'Transaction_Date'])

duplicate_mask = ledger_sorted.duplicated(subset=['Vendor_ID', 'Amount_ZAR'], keep=False)
flagged_invoices = ledger_sorted[duplicate_mask].copy()

# Preview the flagged corporate assets in the notebook
print("--- COOPERATIVE ENTITY COLLUSION RADAR ---")
display(flagged_vendors[['Vendor_ID', 'Vendor_Name', 'Bank_Account', 'Risk_Score']])

print("\n--- HIGH-VELOCITY DUPLICATE INVOICES ---")
display(flagged_invoices[['Invoice_ID', 'Transaction_Date', 'Amount_ZAR', 'Vendor_ID', 'Approver']].head(10))

Creating sample datasets...
Sample data created successfully!
--- COOPERATIVE ENTITY COLLUSION RADAR ---


,Vendor_ID,Vendor_Name,Bank_Account,Risk_Score
0,V001,Company_1,12345678,7
1,V002,Company_2,87654321,4
2,V003,Company_3,12345678,8
3,V004,Company_4,11111111,5
4,V005,Company_5,22222222,7
5,V006,Company_6,12345678,3
6,V007,Company_7,87654321,7
7,V008,Company_8,12345678,8
8,V009,Company_9,11111111,5
9,V010,Company_10,22222222,4



--- HIGH-VELOCITY DUPLICATE INVOICES ---


,Invoice_ID,Transaction_Date,Amount_ZAR,Vendor_ID,Approver
42,INV0043,2026-07-18 10:33:53.784062,1500.0,V002,Manager_C
22,INV0023,2026-08-31 10:33:53.783984,1500.0,V002,Manager_A
36,INV0037,2026-09-06 10:33:53.784038,1500.0,V002,Manager_B
11,INV0012,2026-07-03 10:33:53.783941,3000.0,V003,Manager_C
29,INV0030,2026-07-08 10:33:53.784011,3000.0,V003,Manager_A
39,INV0040,2025-11-19 10:33:53.784049,1500.0,V004,Manager_A
18,INV0019,2026-01-02 10:33:53.783968,1500.0,V004,Manager_B
32,INV0033,2026-02-17 10:33:53.784023,1500.0,V004,Manager_A
44,INV0045,2026-08-07 10:33:53.784070,1500.0,V004,Manager_B
16,INV0017,2026-01-30 10:33:53.783960,3000.0,V007,Manager_C


In [5]:
# --- RUN OPTION B: SPLIT-INVOICE THRESHOLD DETECTOR ---

# 1. Sort ledger chronologically within each Vendor-Approver group
ledger_sorted = ledger_df.sort_values(by=['Vendor_ID', 'Approver', 'Transaction_Date']).copy()

# 2. Calculate hours between consecutive invoices from the same manager to the same vendor
ledger_sorted['Prev_Transaction_Date'] = ledger_sorted.groupby(['Vendor_ID', 'Approver'])['Transaction_Date'].shift(1)
ledger_sorted['Prev_Amount_ZAR'] = ledger_sorted.groupby(['Vendor_ID', 'Approver'])['Amount_ZAR'].shift(1)

ledger_sorted['Time_Window_Hours'] = (ledger_sorted['Transaction_Date'] - ledger_sorted['Prev_Transaction_Date']).dt.total_seconds() / 3600

# 3. Flag transactions to the same vendor by the same manager within 24 hours
ledger_sorted['Split_Invoice_Risk'] = (ledger_sorted['Time_Window_Hours'] <= 24) & (ledger_sorted['Time_Window_Hours'].notna())

# Filter for review
flagged_splits = ledger_sorted[ledger_sorted['Split_Invoice_Risk'] == True]

print(f"--- SPLIT-INVOICE RADAR EXECUTION COMPLETE ---")
print(f"Total potential split-billing exceptions detected: {len(flagged_splits)}")
display(flagged_splits[['Invoice_ID', 'Vendor_ID', 'Approver', 'Transaction_Date', 'Amount_ZAR', 'Time_Window_Hours']].head(10))


--- SPLIT-INVOICE RADAR EXECUTION COMPLETE ---
Total potential split-billing exceptions detected: 0


,Invoice_ID,Vendor_ID,Approver,Transaction_Date,Amount_ZAR,Time_Window_Hours


In [6]:
import numpy as np

# --- 1. INJECT FRAUD ANOMALIES FOR OPTION B TO DETECT ---
# Create explicit back-to-back split invoices breaching an artificial approval limit
fraud_date = pd.to_datetime('2026-03-15 14:00:00')
injected_records = pd.DataFrame([
    {'Invoice_ID': 'INV_FRAUD_A', 'Transaction_Date': fraud_date, 'Amount_ZAR': 420000.0, 'Vendor_ID': 'V003', 'Approver': 'Manager_A'},
    {'Invoice_ID': 'INV_FRAUD_B', 'Transaction_Date': fraud_date + pd.Timedelta(minutes=45), 'Amount_ZAR': 415000.0, 'Vendor_ID': 'V003', 'Approver': 'Manager_A'}
])

# Append anomalies to active ledger and re-run option B
ledger_df = pd.concat([ledger_df, injected_records], ignore_index=True)
ledger_sorted = ledger_df.sort_values(by=['Vendor_ID', 'Approver', 'Transaction_Date']).copy()
ledger_sorted['Prev_Transaction_Date'] = ledger_sorted.groupby(['Vendor_ID', 'Approver'])['Transaction_Date'].shift(1)
ledger_sorted['Time_Window_Hours'] = (ledger_sorted['Transaction_Date'] - ledger_sorted['Prev_Transaction_Date']).dt.total_seconds() / 3600
ledger_sorted['Split_Invoice_Risk'] = (ledger_sorted['Time_Window_Hours'] <= 24) & (ledger_sorted['Time_Window_Hours'].notna())
flagged_splits = ledger_sorted[ledger_sorted['Split_Invoice_Risk'] == True]


# --- 2. RUN ENGINE: OPTION C (BENFORD'S LAW PROFILING) ---
# Extract the first non-zero leading digit of all invoice amounts
ledger_df['First_Digit'] = ledger_df['Amount_ZAR'].astype(str).str.replace('.', '', regex=False).str.lstrip('0').str[0].astype(int)

# Calculate empirical distributions vs Benford's ideal distribution
digit_counts = ledger_df['First_Digit'].value_counts().reindex(range(1, 10), fill_value=0)
empirical_dist = (digit_counts / len(ledger_df)) * 100
benford_dist = [np.log10(1 + 1/d) * 100 for d in range(1, 10)]

# Format results into a clean forensic audit summary profile
benford_analysis = pd.DataFrame({
    'Actual_Frequency_%': empirical_dist.round(2),
    'Benford_Expected_%': [round(b, 2) for b in benford_dist]
})
benford_analysis['Variance_%'] = (benford_analysis['Actual_Frequency_%'] - benford_analysis['Benford_Expected_%']).round(2)

print(f"--- SPLIT-INVOICE RADAR UPDATED ---")
print(f"New split-billing exceptions detected: {len(flagged_splits)}")
display(flagged_splits[['Invoice_ID', 'Vendor_ID', 'Approver', 'Transaction_Date', 'Amount_ZAR', 'Time_Window_Hours']])

print("\n--- BENFORD'S LAW FINANCIAL LEDGER PROFILING ---")
display(benford_analysis)


--- SPLIT-INVOICE RADAR UPDATED ---
New split-billing exceptions detected: 1


,Invoice_ID,Vendor_ID,Approver,Transaction_Date,Amount_ZAR,Time_Window_Hours
51,INV_FRAUD_B,V003,Manager_A,2026-03-15 14:45:00,415000.0,0.75



--- BENFORD'S LAW FINANCIAL LEDGER PROFILING ---


,Actual_Frequency_%,Benford_Expected_%,Variance_%
First_Digit,,,
1,57.69,30.10,27.59
2,9.62,17.61,-7.99
3,28.85,12.49,16.36
4,3.85,9.69,-5.84
5,0.00,7.92,-7.92
6,0.00,6.69,-6.69
7,0.00,5.80,-5.80
8,0.00,5.12,-5.12
9,0.00,4.58,-4.58


In [7]:
# --- PHASE 1 FINAL EXPORT TO PRODUCTION FILES ---

# Save the primary transactional tables
vendors_df.to_csv('final_forensic_vendors.csv', index=False)
ledger_df.to_csv('final_forensic_ledger.csv', index=False)

# Save the processed forensic anomalies summary for direct KPI modeling
benford_analysis.to_csv('final_benford_profile.csv', index=True)

print("--- EXPORT PIPELINE SUCCESSFUL ---")
print("Files generated locally:")
print("1. final_forensic_vendors.csv")
print("2. final_forensic_ledger.csv")
print("3. final_benford_profile.csv")


--- EXPORT PIPELINE SUCCESSFUL ---
Files generated locally:
1. final_forensic_vendors.csv
2. final_forensic_ledger.csv
3. final_benford_profile.csv
